# ZeroHunter — تدريب النموذج وتصدير GGUF

تأكد من اختيار **Runtime → Change runtime type → T4 GPU** قبل تشغيل الخلايا.

In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq cmake libcurl4-openssl-dev build-essential > /dev/null
!pip install -q --no-cache-dir unsloth gguf protobuf sentencepiece
!rm -rf zh_repo && git clone --depth 1 https://github.com/sdxdxa56-design/jdjsjsh.git zh_repo
%cd /content/zh_repo

In [ ]:
# تحديث بيانات الإنترنت قبل البدء وتدريب النموذج مع دمج ملفات التجربة
!python3 -c "from core.knowledge import build_curated_dataset; build_curated_dataset()" || true
!python3 train.py

In [ ]:
import os
import subprocess
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    try:
        os.environ["HF_TOKEN"] = getpass("أدخل HF_TOKEN لرفع النموذج (أو اضغط Enter للتخطي نحو الخادم السريع): ").strip()
    except Exception:
        os.environ["HF_TOKEN"] = ""

if os.environ.get("HF_TOKEN"):
    subprocess.run(["python3", "upload_hf.py"], check=False)
else:
    print("تم تخطي الرفع إلى Hugging Face مؤقتاً، وجاري الانتقال لتشغيل الخادم السريع.")

### خلية 4 (تشغيل فوري في 30 ثانية بدون تدريب): خادم الدردشة السريع (1–3 ثوانٍ للرد)

لتشغيل الدردشة السريعة فوراً: شغّل هذه الخلية وحدها مباشرة (لا تضغط Run all لأن الخلية 2 تبدأ تدريباً يستغرق 25–40 دقيقة).

In [ ]:
import os
import re
import time
import importlib.util
import urllib.request
import subprocess
from pathlib import Path

print("[1/5] تجهيز المستودع وتنظيف أي عمليات قديمة...", flush=True)
if not Path("web_app.py").exists():
    os.chdir("/content")
    if not Path("/content/zh_repo/web_app.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sdxdxa56-design/jdjsjsh.git", "/content/zh_repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/zh_repo", "pull", "--ff-only"], check=False)
    os.chdir("/content/zh_repo")
elif Path(".git").exists():
    subprocess.run(["git", "pull", "--ff-only"], check=False)

subprocess.run(["pkill", "-f", "python3 web_app.py"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "/usr/local/bin/cloudflared"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for old_log in ("web_app.log", "cf.log"):
    if Path(old_log).exists():
        Path(old_log).unlink()

print("[2/5] التحقق من مكتبات التشغيل السريع (عجلات جاهزة بدون ترجمة C++)...", flush=True)
missing_base = [pkg for pkg, mod in (("flask", "flask"), ("requests", "requests"), ("huggingface_hub", "huggingface_hub")) if importlib.util.find_spec(mod) is None]
if missing_base:
    subprocess.run(["pip", "install", "-q", *missing_base], check=False)

if importlib.util.find_spec("llama_cpp") is None:
    installed_llama = False
    for backend_name, wheel_url in (
        ("GPU CUDA 12.4", "https://abetlen.github.io/llama-cpp-python/whl/cu124"),
        ("GPU CUDA 12.1", "https://abetlen.github.io/llama-cpp-python/whl/cu121"),
        ("CPU Fast Wheel", "https://abetlen.github.io/llama-cpp-python/whl/cpu"),
    ):
        print(f"   جاري تجربة العجلة الجاهزة: {backend_name}...", flush=True)
        res = subprocess.run(
            ["pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", wheel_url, "llama-cpp-python"],
            check=False,
        )
        if res.returncode == 0 and importlib.util.find_spec("llama_cpp") is not None:
            print(f"   تم تثبيت llama-cpp-python ({backend_name}) بنجاح!", flush=True)
            installed_llama = True
            break
    if not installed_llama:
        raise RuntimeError("تعذر العثور على عجلة جاهزة (wheel) متوافقة لـ llama-cpp-python على هذه البيئة.")
else:
    print("   مكتبة llama-cpp-python مثبتة مسبقاً.", flush=True)

print("[3/5] تجهيز نفق Cloudflare...", flush=True)
if not Path("/usr/local/bin/cloudflared").exists():
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

print("[4/5] تجهيز أحدث نموذج مدرب (من GitHub / Hugging Face)...", flush=True)
from huggingface_hub import hf_hub_download

model_path = ""
gguf_candidates = sorted(Path(".").rglob("*.gguf"), key=lambda p: p.stat().st_size, reverse=True)
if gguf_candidates:
    model_path = str(gguf_candidates[0].resolve())
    print(f"   تم العثور على نموذج محلي: {Path(model_path).name}", flush=True)
else:
    try:
        print(f"   جاري محاولة تحميل أحدث نموذج مدرب من sdxdxa56/zerohunter-qwen-gguf (zerohunter-7b-q5_k_m.gguf)...", flush=True)
        model_path = hf_hub_download(repo_id="sdxdxa56/zerohunter-qwen-gguf", filename="zerohunter-7b-q5_k_m.gguf", token=os.environ.get("HF_TOKEN") or None)
        print("   تم تحميل نموذجك المخصص بنجاح!", flush=True)
    except Exception:
        print("   لم يُرفع النموذج المخصص بعد، جاري جلب نموذج Qwen2.5-Coder-1.5B السريع (~986 MB)...", flush=True)
        model_path = hf_hub_download(
            repo_id="Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF",
            filename="qwen2.5-coder-1.5b-instruct-q4_k_m.gguf",
        )

print("[5/5] تشغيل الخادم المحلي وفتح الرابط العام (مع فصل الجلسة لإظهار علامة الصح)...", flush=True)
env = os.environ.copy()
env["ZEROHUNTER_LOCAL_MODEL"] = model_path
env["PYTHONUNBUFFERED"] = "1"
with open("web_app.log", "w") as web_out:
    web_proc = subprocess.Popen(
        ["python3", "web_app.py"],
        env=env,
        stdin=subprocess.DEVNULL,
        stdout=web_out,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        close_fds=True,
    )

server_ready = False
for _ in range(25):
    if web_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:5000/", timeout=2) as resp:
            if resp.status < 500:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

with open("cf.log", "w") as cf_out:
    cf_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
        stdin=subprocess.DEVNULL,
        stdout=cf_out,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        close_fds=True,
    )

public_url = ""
for _ in range(35):
    if cf_proc.poll() is not None:
        break
    time.sleep(1)
    if Path("cf.log").exists():
        log_txt = Path("cf.log").read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", log_txt)
        if match:
            public_url = match.group(0)
            break

if public_url and server_ready:
    print("\n✅ خادم ZeroHunter السريع يعمل الآن واكتملت الخلية بنجاح!", flush=True)
    print("انسخ هذا الرابط وضعه في AI Studio Secrets باسم ZEROHUNTER_LOCAL_URL:", flush=True)
    print(public_url, flush=True)
else:
    print("تعذر إتمام التشغيل السريع:", {"server_ready": server_ready, "public_url": public_url}, flush=True)
    if Path("web_app.log").exists():
        print("=== web_app.log ===", flush=True)
        print(Path("web_app.log").read_text(encoding="utf-8", errors="ignore")[-1000:], flush=True)
    if Path("cf.log").exists():
        print("=== cf.log ===", flush=True)
        print(Path("cf.log").read_text(encoding="utf-8", errors="ignore")[-1000:], flush=True)